#Document Question Answering System using Retrieval-Augmented Generative (RAG)
#Step 1 : Objective :
This project implements a Retrieval-Augmented Generation (RAG) system capable of answering user questions based on custom documents such as PDFs.

Instead of relying solely on Large Language Model's internal knowledge, the system retrieves relevant document chunks using semantic search and then generates grounded responses.

#Workflow

*   Document Loading
*   Text Extraction
*   Text Chunking
*   Embedding Generation
*   Vector Database (FAISS)
*   Similarity Retrieval
*   Prompt Augmentation
*   Answer Generation using Gemini
*   Validation and Evaluation







#Step 2 - Install Required Libraries

In [1]:
!pip install -U langchain langchain-community langchain-google-genai langchain-text-splitters faiss-cpu sentence-transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.8
    Uninstalling langchain-core-1.4

In [2]:
!pip install -q langgraph

In [3]:
import langchain
import langchain_community
import langchain_google_genai
print(langchain.__version__)
print(langchain_community.__version__)
print(langchain_google_genai.__version__)

/tmp/ipykernel_677/2204395959.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


1.3.13
0.4.2
4.2.7


#Step 3 - Import Libraries

In [4]:
import os
import warnings
warnings.filterwarnings("ignore")
from google.colab import userdata
# PDF Loader
from langchain_community.document_loaders import PyPDFLoader
# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
# Vector Store
from langchain_community.vectorstores import FAISS
print("✅ Imports Successful")

✅ Imports Successful


#Step 4 - Load the API

In [ ]:
os.environ["GOOGLE_API_KEY"] = "Your API KEY"
print("API Loaded Successfully")

API Loaded Successfully


#Step 5 - Upload Your PDF

In [6]:
from google.colab import files
uploaded = files.upload()

Saving attention.pdf to attention.pdf


#Step 6 - Load the PDF
Step 6.1: Document Loading
Load the uploaded PDF using LangChain's PyPDFLoader.

In [7]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path = list(uploaded.keys())[0]
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print("PDF Loaded Successfully")
print("Total Pages:", len(documents))

PDF Loaded Successfully
Total Pages: 15


Observation
The PDF document was successfully loaded using LangChain's `PyPDFLoader`.

Each page was extracted as an individual document object, making it suitable for subsequent preprocessing stages such as chunking and embedding generation.

The successful extraction confirms that the document ingestion stage of the RAG pipeline has been completed.

#Step 7 - View the First Page

In [8]:
print(documents[0].page_content[:1000])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exp

#Step 8 - Text Chunking

Text Chunking
The extracted document is divided into smaller overlapping chunks.
Chunking improves semantic retrieval by preserving context while keeping each text segment manageable for embedding generation.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print("Total Chunks:", len(chunks))

Total Chunks: 103


#Verify Chunk

In [10]:
print(chunks[0].page_content)

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu


Observation

The uploaded document was successfully divided into **103 overlapping chunks**.

Chunking ensures that long documents are broken into manageable semantic units while preserving contextual continuity through overlapping regions.

Using a chunk size of **500 characters** with **100-character overlap** helps improve retrieval accuracy by reducing information loss at chunk boundaries.

#Step 9 - Create Embeddings

In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001"
)
print("Embedding Model Loaded")

Embedding Model Loaded


#Step 10 - GoogleGenerativeAIEmbeddings

In [12]:
from google import genai
import os
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
print("Gemini Client Connected")

Gemini Client Connected


#Step 11 - Create Embeddings

In [13]:
import time
texts = [doc.page_content for doc in chunks]
embeddings = []
for text in texts:
    while True:
        try:
            response = client.models.embed_content(
                model="models/gemini-embedding-001",
                contents=text
            )
            embeddings.append(response.embeddings[0].values)
            break
        except Exception as e:
            print("Retrying...", e)
            time.sleep(5)
print("Total Embeddings:", len(embeddings))

Retrying... 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 56.622820235s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-

Observation :
Each document chunk is transformed into a dense numerical vector using the **Gemini Embedding Model (`gemini-embedding-001`)**.

Embeddings capture the semantic meaning of text rather than simple keyword matching. This enables the retrieval system to identify contextually relevant document chunks when answering user queries.





#Step 12 - FAISS Vector Database

After generating embeddings, all vectors are stored in a **FAISS (Facebook AI Similarity Search)** index.

FAISS provides efficient similarity search over high-dimensional vectors, enabling the retrieval of document chunks that are semantically closest to the user's query.

The vector database forms the retrieval component of the RAG pipeline.

In [14]:
import faiss
import numpy as np
embedding_matrix = np.array(embeddings, dtype=np.float32)
dimension = embedding_matrix.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)
print("FAISS Index Created Successfully")
print("Embedding Dimension:", dimension)
print("Total Stored Vectors:", index.ntotal)

FAISS Index Created Successfully
Embedding Dimension: 3072
Total Stored Vectors: 103


Observation

The FAISS vector database was successfully constructed using the generated document embeddings. Each document chunk was represented as a high-dimensional vector and indexed within the FAISS search structure to enable efficient semantic similarity retrieval.


*    **Embedding Model:** Gemini Embedding 001
*   Embedding Dimension: **3072**
*   Indexed Vectors: **103**
*   Vector Database: **FAISS IndexFlatL2**

The successful creation of the FAISS index confirms that the document preprocessing and embedding generation stages have been completed successfully. The vector database now serves as the knowledge repository for the Retrieval-Augmented Generation (RAG) pipeline.

When a user submits a query, it will first be converted into an embedding vector using the same embedding model. FAISS will then perform a similarity search to retrieve the most contextually relevant document chunks, which will subsequently be provided to the language model for generating accurate and grounded responses.

#Step 13 – Context Retrieval

In [15]:
def retrieve(query, k=3):
    response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=query
    )
    query_embedding = np.array(
        [response.embeddings[0].values],
        dtype=np.float32
    )
    distances, indices = index.search(query_embedding, k)
    retrieved_docs = [chunks[i] for i in indices[0]]
    return retrieved_docs

#Step 14 – Test the Retriever

In [16]:
query = "What is the Transformer architecture?"
retrieved_docs = retrieve(query)
print(f"Retrieved {len(retrieved_docs)} document chunks.\n")
for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"Retrieved Chunk {i}")
    print("=" * 80)
    print(doc.page_content[:800])
    print()

Retrieved 3 document chunks.

Retrieved Chunk 1
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-

Retrieved Chunk 2
To the best of our knowledge, however, the Transformer is the first transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
self-attention and discuss its advantages over models such as [17, 18] and [9].
3 Model Architecture
Most competitive neural sequence transduction model

Observation :
The retrieval module successfully identified the most relevant document chunks corresponding to the user's query **"What is the Transformer architecture?"** using semantic similarity search.

The retrieved chunks accurately describe the Transformer architecture, including its encoder-decoder structure, multi-head self-attention mechanism, and its advantages over recurrent and convolutional neural networks.


*   **Query:** *What is the Transformer architecture?*
*   **Retrieval Method:** Semantic Similarity Search
*   **Vector Database:** FAISS (IndexFlatL2)
*   **Top Retrieved Chunks:** 3
*   **Embedding Model:** Gemini Embedding 001


The retrieved chunks are highly relevant to the user's query, demonstrating that the embedding model and FAISS vector database are functioning correctly. Instead of relying on keyword matching, the system retrieves context based on semantic meaning, ensuring that the subsequent language model receives accurate and informative context for grounded answer generation.

This successfully completes the **retrieval phase** of the Retrieval-Augmented Generation (RAG) pipeline.




#Step 15 – Initialize Gemini Model

In [17]:
response = client.models.generate_content(
    model="models/gemini-3.5-flash",
    contents="Say Hello"
)
print(response.text)

Hello! How can I help you today?


#Step 16 – Create RAG Prompt

In [18]:
def create_prompt(query, retrieved_docs):
  context = "\n\n".join([doc.page_content for doc in retrieved_docs])
  prompt = f"""
You are an AI assistant for Document Question Answering.
Answer the question ONLY using the information provided in the context.
If the answer is not available in the context, reply:
'I could not find the answer in the provided document.'
Context:
{context}

Question:
{query}

Answer:
"""
  return prompt

#Step 17 – Generate Grounded Answer

In [19]:
query = "What is the Transformer architecture?"
# Retrieve relevant chunks
retrieved_docs = retrieve(query)
# Create prompt
prompt = create_prompt(query, retrieved_docs)
# Generate grounded answer
response = client.models.generate_content(
    model="models/gemini-3.5-flash",
    contents=prompt
)
print(response.text)

Based on the provided document, the Transformer is a network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. 

Its overall architecture uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. The encoder is composed of a stack of N = 6 identical layers, where each layer has two sub-layers:
1. A multi-head self-attention mechanism.
2. A simple, position- [text cuts off here].


Observation
The Retrieval-Augmented Generation (RAG) pipeline successfully generated a grounded response using the retrieved document context.

The language model answered the user's question by incorporating only the relevant information retrieved from the FAISS vector database rather than relying solely on its pre-trained knowledge.


*   **Query:** *What is the Transformer architecture?*
*   **Language Model:** Gemini 3.5 Flash
*   **Retrieved Context:** Top 3 relevant document chunks
*   **Response Type:** Context-Grounded Answer

The generated response accurately summarizes the Transformer architecture, including its encoder-decoder design, multi-head self-attention mechanism, and the elimination of recurrent and convolutional networks.

This demonstrates that the retrieval component successfully supplied relevant context to the language model, enabling the generation of accurate and document-specific answers. The successful integration of retrieval and generation confirms the completion of the end-to-end Document Question Answering (RAG) system.


#Step 18 – Validation with Multiple Questions

In [21]:
import time

test_queries = [
    "What is the Transformer architecture?",
    "What is self-attention?",
    "How many encoder layers are used?",
    "Why is the Transformer better than RNNs?",
    "What are the two sub-layers in each encoder layer?"
]

for i, query in enumerate(test_queries, start=1):
    print("=" * 100)
    print(f"Test Query {i}: {query}")
    docs = retrieve(query)
    prompt = create_prompt(query, docs)
    while True:
        try:
            response = client.models.generate_content(
                model="models/gemini-3.5-flash",
                contents=prompt
            )
            break
        except Exception as e:
            print("Rate limit reached. Waiting 15 seconds...")
            time.sleep(15)
    print("\nGenerated Answer:\n")
    print(response.text)
    print("\n")

Test Query 1: What is the Transformer architecture?

Generated Answer:

Based on the provided context, the Transformer is a network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. 

Its architecture is characterized by:
* An encoder-decoder structure.
* The use of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.
* An encoder composed of a stack of $N = 6$ identical layers, where each layer has two sub-layers: a multi-head self-attention mechanism and a second position-based sub-layer. 
* Relying entirely on self-attention to compute representations of its input and output without using sequence-aligned RNNs or convolution.


Test Query 2: What is self-attention?

Generated Answer:

Based on the provided document, self-attention (sometimes called intra-attention) is an attention mechanism relating different positions of a single sequence in order to compute a representation of the seq

Observation
The Document Question Answering System was evaluated using multiple domain-specific questions related to the uploaded research paper.

For every query, the retrieval module successfully identified relevant document chunks, and the Gemini language model generated grounded responses using only the retrieved context.

*   Total Test Queries: **5**
*   Retrieval Method: **Semantic Similarity Search**
*   Context Source: **FAISS Vector Database**
*   Language Model: **Gemini 3.5 Flash**

The successful responses to multiple queries demonstrate that the retrieval mechanism consistently provides relevant contextual information to the language model. This validates the effectiveness of the Retrieval-Augmented Generation (RAG) pipeline for document-grounded question answering.

#Step 19 – System Metrics Report

In [22]:
print("=" * 60)
print("SYSTEM METRICS REPORT")
print("=" * 60)
print(f"PDF Pages               : {len(documents)}")
print(f"Document Chunks         : {len(chunks)}")
print(f"Chunk Size              : 500")
print(f"Chunk Overlap           : 100")
print(f"Embedding Model         : Gemini Embedding 001")
print(f"Embedding Dimension     : {dimension}")
print(f"Vector Database         : FAISS (IndexFlatL2)")
print(f"Language Model          : Gemini 3.5 Flash")
print(f"Retrieved Chunks (Top-K): 3")

SYSTEM METRICS REPORT
PDF Pages               : 15
Document Chunks         : 103
Chunk Size              : 500
Chunk Overlap           : 100
Embedding Model         : Gemini Embedding 001
Embedding Dimension     : 3072
Vector Database         : FAISS (IndexFlatL2)
Language Model          : Gemini 3.5 Flash
Retrieved Chunks (Top-K): 3


Observation

The system configuration and implementation parameters are summarized below.


*   Document Pages: **15**
*   Document Chunks: **103**
*   Chunk Size: **500**
*   Chunk Overlap: **100**
*   Embedding Model: **Gemini Embedding 001**
*   Embedding Dimension: **3072**
*   Vector Store: **FAISS (IndexFlatL2)**
*   Language Model: **Gemini 3.5 Flash**
*   Top-k Retrieval: **3**

The selected configuration provides a balance between retrieval accuracy and computational efficiency. The generated embeddings and FAISS indexing enable efficient semantic search, while Gemini 3.5 Flash produces grounded responses using the retrieved document context.


#Step 20 – Chunking Experiment


In [23]:
experiment_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
experiment_chunks = experiment_splitter.split_documents(documents)
print("Original Chunks :", len(chunks))
print("Experimental Chunks :", len(experiment_chunks))

Original Chunks : 103
Experimental Chunks : 164


Observation


An additional experiment was performed by reducing the chunk size from **500** to **300** characters.

Smaller chunks increase the number of document segments, which may improve retrieval precision for highly specific queries. However, they also increase storage requirements and retrieval latency.


*   Configuration | Chunk Size | Overlap |
*   | Default | 500 | 100 |
*   | Experimental | 300 | 50 |

*   The experiment demonstrates that chunk size significantly influences retrieval performance. Larger chunks preserve more context, while smaller chunks provide more fine-grained retrieval. Selecting an appropriate chunk size depends on the document characteristics and application requirements.


#Conclusion

A complete Retrieval-Augmented Generation (RAG) based Document Question Answering System was successfully implemented using LangChain, Google Gemini, and FAISS.

The system performs document ingestion, text chunking, embedding generation, semantic retrieval, and grounded response generation. By retrieving relevant document context before invoking the language model, the system produces accurate, context-aware, and document-specific responses while reducing hallucinations.

The experimental evaluation confirms that the proposed pipeline effectively supports question answering over custom documents and can be extended for enterprise knowledge assistants, research document analysis, and intelligent document search applications.